# BioRydberg — QoolQit Contest Notebook

**Neutral-Atom Quantum Simulation of Biological Energy-Transport-Inspired Networks**

This notebook intentionally compares two different models:

- a biological single-excitation hopping model;
- a driven Rydberg neutral-atom model implemented with QoolQit.

The goal is to investigate similarity in **spatial population patterns**, not to claim exact Hamiltonian equivalence.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

# Make src/ importable whether the notebook is launched from the repo root
# or from inside the notebooks directory.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.biological_model import three_site_example
from src.neutral_atom_mapping import couplings_to_qoolqit_coordinates
from src.qoolqit_simulation import run_occupation_dynamics
from src.analysis import normalize_rows, spatial_profile_rmse
from src.visualization import plot_site_populations, plot_network, plot_comparison

print("Project root:", ROOT)


## 1. Biological reference model

We use a three-site excitonic Hamiltonian:

\[
H_{bio}=\sum_i E_i|i\rangle\langle i|+
\sum_{i\ne j}J_{ij}|i\rangle\langle j|.
\]

The excitation starts localized on Site 1.


In [ ]:
H, bio_times, bio_populations = three_site_example()
print(H)

fig = plot_site_populations(
    bio_times,
    bio_populations,
    "Three-site biological reference"
)
plt.show()


## 2. Map coupling structure into neutral-atom geometry

QoolQit's dimensionless Rydberg interaction scales as approximately
\(1/r_{ij}^6\). We therefore use stronger biological graph connections as a
heuristic for placing atoms relatively closer together.


In [ ]:
couplings = H.copy()
np.fill_diagonal(couplings, 0.0)

coordinates = couplings_to_qoolqit_coordinates(couplings)
print(coordinates)

fig = plot_network(
    coordinates,
    coupling_matrix=couplings,
    title="Coupling-informed QoolQit geometry"
)
plt.show()


## 3. QoolQit neutral-atom program

Workflow:

**Register → Drive → QuantumProgram → Compile → LocalEmulator → Occupation**


In [ ]:
qrun = run_occupation_dynamics(
    coordinates=coordinates,
    duration=10.0,
    amplitude=0.8,
    detuning=0.0,
    n_times=31,
)

print("QoolQit 1/r^6 interaction matrix:")
print(qrun.interaction_matrix)

fig = plot_site_populations(
    qrun.relative_times,
    qrun.occupations,
    "QoolQit Rydberg occupation dynamics",
    ylabel="Rydberg occupation <n_i>",
)
plt.show()


## 4. Compare spatial patterns

The biological model conserves exactly one excitation. The driven Rydberg
model does not. We therefore normalize each QoolQit time slice before comparing
the *spatial distribution*.


In [ ]:
qoolqit_spatial = normalize_rows(qrun.occupations)

rmse = spatial_profile_rmse(
    bio_populations,
    qrun.occupations,
)

print("Spatial-profile RMSE:", rmse)

fig = plot_comparison(
    biological=bio_populations,
    rydberg_normalized=qoolqit_spatial,
)
plt.show()


## 5. Interpretation

A lower spatial-profile RMSE means the neutral-atom occupation pattern more
closely resembles the reference biological population pattern on normalized
relative time.

It **does not** mean the biological and Rydberg Hamiltonians are identical.

### Competition extension

Sweep over:

- drive amplitude \(\Omega\)
- detuning \(\delta\)
- duration
- geometry

and ask:

> Which QoolQit controls best reproduce the desired spatial transport pattern?
